# MTG Premodern - Scraping de Metadata

Este notebook scrapea toda la metadata de torneos Premodern desde tcdecks.net:
- Torneos (nombre, fecha, cantidad de jugadores, fuente)
- Mazos (jugador, arquetipo, posición, torneo)

**Fuente:** `results.php?format=Premodern` (~1,262 páginas, ~37,853 mazos)

**Tiempo estimado:** ~32 minutos

In [ ]:
# Celda 1: Instalar dependencias
!pip install -q requests beautifulsoup4 lxml tqdm

In [ ]:
# Celda 2: Imports y configuración
import sqlite3
import time
import re
import logging
from datetime import datetime
from urllib.parse import parse_qs, urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuración
BASE_URL = "https://www.tcdecks.net"
RESULTS_URL = f"{BASE_URL}/results.php"
DB_FILE = "premodern.db"
DELAY = 1.5  # segundos entre requests

print("Configuración lista.")

In [ ]:
# Celda 3: Funciones utilitarias

def parse_date(date_str):
    """Parse DD/MM/YYYY to ISO YYYY-MM-DD."""
    date_str = date_str.strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    raise ValueError(f"Cannot parse date: {date_str}")


def parse_position(text):
    """Parse '5 of 138' into (position, total_players)."""
    match = re.search(r"(\d+)\s+of\s+(\d+)", text)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None


def extract_deck_ids(href):
    """Extract tournament_id and deck_id from deck.php URL."""
    parsed = urlparse(href)
    params = parse_qs(parsed.query)
    tid = int(params["id"][0]) if "id" in params else None
    did = int(params["iddeck"][0]) if "iddeck" in params else None
    return tid, did


def detect_source(name):
    """Detect tournament source from name."""
    lower = name.lower()
    if any(kw in lower for kw in ("mtgo", "league", "challenge", "preliminary", "showcase")):
        return "mol"
    if "webcam" in lower:
        return "webcam"
    return "paper"


print("Funciones utilitarias cargadas.")

In [ ]:
# Celda 4: Crear base de datos SQLite

SCHEMA = """
PRAGMA journal_mode=WAL;
PRAGMA foreign_keys=ON;

CREATE TABLE IF NOT EXISTS tournaments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    date DATE NOT NULL,
    player_count INTEGER,
    source TEXT CHECK(source IN ('paper', 'webcam', 'mol', 'unknown')) DEFAULT 'unknown',
    url TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_tournaments_date ON tournaments(date);
CREATE INDEX IF NOT EXISTS idx_tournaments_source ON tournaments(source);

CREATE TABLE IF NOT EXISTS decks (
    id INTEGER PRIMARY KEY,
    tournament_id INTEGER NOT NULL REFERENCES tournaments(id),
    player_name TEXT NOT NULL,
    archetype TEXT NOT NULL,
    archetype_url TEXT,
    position INTEGER,
    total_players INTEGER,
    date DATE NOT NULL,
    cards_scraped BOOLEAN DEFAULT 0,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_decks_tournament ON decks(tournament_id);
CREATE INDEX IF NOT EXISTS idx_decks_archetype ON decks(archetype);
CREATE INDEX IF NOT EXISTS idx_decks_player ON decks(player_name);
CREATE INDEX IF NOT EXISTS idx_decks_date ON decks(date);
CREATE INDEX IF NOT EXISTS idx_decks_unscraped ON decks(cards_scraped) WHERE cards_scraped = 0;

CREATE TABLE IF NOT EXISTS cards (
    name TEXT PRIMARY KEY
);

CREATE TABLE IF NOT EXISTS deck_cards (
    deck_id INTEGER NOT NULL REFERENCES decks(id),
    card_name TEXT NOT NULL REFERENCES cards(name),
    quantity INTEGER NOT NULL CHECK(quantity > 0),
    is_sideboard BOOLEAN NOT NULL DEFAULT 0,
    PRIMARY KEY (deck_id, card_name, is_sideboard)
);

CREATE INDEX IF NOT EXISTS idx_deck_cards_card ON deck_cards(card_name);
CREATE INDEX IF NOT EXISTS idx_deck_cards_deck ON deck_cards(deck_id);

CREATE TABLE IF NOT EXISTS scrape_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    phase TEXT NOT NULL,
    last_page INTEGER,
    total_new_records INTEGER DEFAULT 0,
    started_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    completed_at TIMESTAMP,
    status TEXT CHECK(status IN ('running', 'completed', 'failed')) DEFAULT 'running'
);
"""

conn = sqlite3.connect(DB_FILE)
conn.executescript(SCHEMA)
conn.commit()
print(f"Base de datos '{DB_FILE}' creada con schema completo.")

In [ ]:
# Celda 5: Configurar sesión HTTP con reintentos

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,es;q=0.8",
})

retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

print("Sesión HTTP configurada con reintentos automáticos.")

In [ ]:
# Celda 6: Función para parsear una página de results.php
#
# Estructura HTML real de tcdecks.net/results.php:
#   <tr>  (sin clase, plain tr)
#     <td data-th="Archetype">   <a href="deck.php?id=T&iddeck=D">Archetype</a>
#     <td data-th="Format">      <a href="...">Premodern</a>
#     <td data-th="Player">      <a href="...">PlayerName</a>
#     <td data-th="Tournament Name"> <a href="deck.php?id=T">Tournament - Date</a>
#     <td data-th="Position">    X of Y
#     <td data-th="Date">        DD/MM/YYYY

BASE_SITE = "https://www.tcdecks.net"

def scrape_results_page(page_num):
    """Scrape a single page of results.php and return parsed data."""
    response = session.get(RESULTS_URL, params={
        "format": "Premodern",
        "src": "all",
        "page": page_num,
    }, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")
    rows = soup.find_all("tr")
    results = []

    for row in rows:
        try:
            cells = row.find_all("td")
            if len(cells) < 6:
                continue

            # Verificar que es una fila de datos (tiene data-th)
            if not cells[0].get("data-th"):
                continue

            # Celda 0: Archetype + link con iddeck
            arch_link = cells[0].find("a", href=True)
            if not arch_link:
                continue
            archetype = arch_link.get_text(strip=True)
            href = arch_link.get("href", "")

            # Extraer IDs del link deck.php?id=T&iddeck=D
            tournament_id, deck_id = extract_deck_ids(href)
            if not tournament_id or not deck_id:
                continue

            # Celda 2: Player
            player_link = cells[2].find("a")
            player_name = player_link.get_text(strip=True) if player_link else "Unknown"

            # Celda 3: Tournament Name
            tourn_link = cells[3].find("a")
            tournament_name = tourn_link.get_text(strip=True) if tourn_link else "Unknown"

            # Celda 4: Position ("X of Y")
            position_text = cells[4].get_text(strip=True)
            position, total_players = parse_position(position_text)

            # Celda 5: Date (DD/MM/YYYY)
            date_text = cells[5].get_text(strip=True)
            if not date_text:
                continue
            parsed_date = parse_date(date_text)

            source = detect_source(tournament_name)

            # Construir URLs
            tournament_url = f"{BASE_SITE}/deck.php?id={tournament_id}"
            archetype_url = f"{BASE_SITE}/archetype.php?archetype={archetype}&format=Premodern"

            results.append({
                "tournament_id": tournament_id,
                "tournament_name": tournament_name,
                "tournament_url": tournament_url,
                "date": parsed_date,
                "player_count": total_players,
                "source": source,
                "deck_id": deck_id,
                "player_name": player_name,
                "archetype": archetype,
                "archetype_url": archetype_url,
                "position": position,
                "total_players": total_players,
            })

        except Exception as e:
            logger.warning(f"Error parsing row: {e}")
            continue

    return results


def get_total_pages():
    """Detect total pages from first page."""
    response = session.get(RESULTS_URL, params={
        "format": "Premodern", "src": "all", "page": 1
    }, timeout=30)
    soup = BeautifulSoup(response.text, "lxml")
    max_page = 1
    for link in soup.select("a[href*='page=']"):
        match = re.search(r"page=(\d+)", link.get("href", ""))
        if match:
            max_page = max(max_page, int(match.group(1)))
    return max_page


print("Funciones de scraping cargadas.")

In [ ]:
# Celda 7: TEST - Validar scraper con la primera página
# Ejecutá esta celda para verificar que el parser funciona antes de lanzar el scraping completo.

print("Testeando scraper con página 1...")
test_results = scrape_results_page(1)

print(f"\nFilas parseadas: {len(test_results)}")
if len(test_results) == 0:
    print("⚠️  ERROR: No se parseó ninguna fila. El HTML del sitio puede haber cambiado.")
    print("Revisá la estructura HTML manualmente antes de continuar.")
else:
    print(f"✅ Parser OK - {len(test_results)} registros encontrados\n")
    # Mostrar los primeros 5 como tabla
    import pandas as pd
    df_test = pd.DataFrame(test_results[:5])
    cols = ["archetype", "player_name", "tournament_name", "position", "total_players", "date", "source", "tournament_id", "deck_id"]
    print(df_test[cols].to_string(index=False))

    # Mostrar URLs de ejemplo
    print(f"\n--- URLs de ejemplo (fila 1) ---")
    print(f"  Tournament URL:  {test_results[0]['tournament_url']}")
    print(f"  Archetype URL:   {test_results[0]['archetype_url']}")

    # Validaciones básicas
    print(f"\n--- Checks ---")
    print(f"  Arquetipos únicos: {len(set(r['archetype'] for r in test_results))}")
    print(f"  Jugadores únicos:  {len(set(r['player_name'] for r in test_results))}")
    print(f"  Torneos únicos:    {len(set(r['tournament_id'] for r in test_results))}")
    print(f"  Todos tienen deck_id:       {'✅' if all(r['deck_id'] for r in test_results) else '❌'}")
    print(f"  Todos tienen tournament_id: {'✅' if all(r['tournament_id'] for r in test_results) else '❌'}")
    print(f"  Todos tienen fecha:         {'✅' if all(r['date'] for r in test_results) else '❌'}")
    print(f"  Todos tienen posición:      {'✅' if all(r['position'] for r in test_results) else '❌'}")
    print(f"  Todos tienen tournament_url: {'✅' if all(r['tournament_url'] for r in test_results) else '❌'}")
    print(f"  Todos tienen archetype_url:  {'✅' if all(r['archetype_url'] for r in test_results) else '❌'}")

    if len(test_results) == 30:
        print(f"\n🚀 Todo OK. Podés continuar con el scraping completo (celdas siguientes).")

In [ ]:
# Celda 8: Detectar total de páginas

total_pages = get_total_pages()
print(f"Total de páginas a scrapear: {total_pages}")
print(f"Registros estimados: ~{total_pages * 30:,}")
print(f"Tiempo estimado: ~{total_pages * DELAY / 60:.0f} minutos")

In [ ]:
# Celda 9: Ejecutar scraping completo

# Registrar inicio en scrape_log
conn.execute(
    "INSERT INTO scrape_log (phase, status) VALUES ('metadata', 'running')"
)
conn.commit()
log_id = conn.execute("SELECT last_insert_rowid()").fetchone()[0]

new_decks = 0
errors = 0
existing_deck_ids = set(
    row[0] for row in conn.execute("SELECT id FROM decks").fetchall()
)

print(f"Decks existentes en DB: {len(existing_deck_ids)}")
print(f"Iniciando scraping de {total_pages} páginas...\n")

pbar = tqdm(range(1, total_pages + 1), desc="Scraping", unit="página")

for page in pbar:
    try:
        results = scrape_results_page(page)

        page_new = 0
        for r in results:
            # Upsert tournament (con URL)
            conn.execute(
                """INSERT INTO tournaments (id, name, date, player_count, source, url)
                   VALUES (?, ?, ?, ?, ?, ?)
                   ON CONFLICT(id) DO UPDATE SET
                     player_count = MAX(tournaments.player_count, excluded.player_count),
                     source = CASE WHEN tournaments.source = 'unknown'
                                   THEN excluded.source ELSE tournaments.source END,
                     url = COALESCE(tournaments.url, excluded.url)""",
                (r["tournament_id"], r["tournament_name"], r["date"],
                 r["player_count"], r["source"], r["tournament_url"]),
            )

            # Insert deck (con archetype_url)
            if r["deck_id"] not in existing_deck_ids:
                try:
                    conn.execute(
                        """INSERT OR IGNORE INTO decks
                           (id, tournament_id, player_name, archetype, archetype_url,
                            position, total_players, date)
                           VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                        (r["deck_id"], r["tournament_id"], r["player_name"],
                         r["archetype"], r["archetype_url"],
                         r["position"], r["total_players"], r["date"]),
                    )
                    if conn.execute("SELECT changes()").fetchone()[0] > 0:
                        existing_deck_ids.add(r["deck_id"])
                        page_new += 1
                        new_decks += 1
                except sqlite3.IntegrityError:
                    pass

        # Commit cada 10 páginas
        if page % 10 == 0:
            conn.commit()

        pbar.set_postfix({
            "nuevos": new_decks,
            "página_actual": f"{len(results)} rows",
            "errores": errors,
        })

        time.sleep(DELAY)

    except Exception as e:
        errors += 1
        logger.error(f"Error en página {page}: {e}")
        time.sleep(5)
        continue

# Commit final
conn.execute(
    """UPDATE scrape_log SET
       last_page = ?, total_new_records = ?,
       completed_at = CURRENT_TIMESTAMP, status = 'completed'
       WHERE id = ?""",
    (total_pages, new_decks, log_id),
)
conn.commit()

print(f"\n{'='*50}")
print(f"Scraping completado!")
print(f"Nuevos mazos insertados: {new_decks:,}")
print(f"Errores: {errors}")

In [ ]:
# Celda 9: Validación

import pandas as pd

print("=" * 50)
print("VALIDACIÓN DE DATOS")
print("=" * 50)

# Conteos
total_tournaments = conn.execute("SELECT COUNT(*) FROM tournaments").fetchone()[0]
total_decks = conn.execute("SELECT COUNT(*) FROM decks").fetchone()[0]
print(f"\nTorneos: {total_tournaments:,}")
print(f"Mazos: {total_decks:,}")

# Rango de fechas
date_range = conn.execute(
    "SELECT MIN(date), MAX(date) FROM tournaments"
).fetchone()
print(f"Rango de fechas: {date_range[0]} a {date_range[1]}")

# Top arquetipos
print("\nTop 15 arquetipos:")
df_arch = pd.read_sql_query(
    """SELECT archetype, COUNT(*) as count,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM decks), 1) as pct
       FROM decks GROUP BY archetype ORDER BY count DESC LIMIT 15""",
    conn
)
print(df_arch.to_string(index=False))

# Distribución por source
print("\nDistribución por fuente:")
df_source = pd.read_sql_query(
    """SELECT source, COUNT(*) as tournaments,
       (SELECT COUNT(*) FROM decks d
        JOIN tournaments t ON d.tournament_id = t.id
        WHERE t.source = tournaments.source) as decks
       FROM tournaments GROUP BY source""",
    conn
)
print(df_source.to_string(index=False))

# Sample de datos
print("\nÚltimos 5 torneos:")
df_recent = pd.read_sql_query(
    "SELECT id, name, date, player_count, source FROM tournaments ORDER BY date DESC LIMIT 5",
    conn
)
print(df_recent.to_string(index=False))

In [ ]:
# Celda 10: Descargar la base de datos

conn.close()

try:
    from google.colab import files
    files.download(DB_FILE)
    print(f"Descargando {DB_FILE}...")
except ImportError:
    print(f"No estás en Colab. El archivo está en: {DB_FILE}")
    print("Copialo manualmente a tu proyecto local en db/premodern.db")